In [ ]:
from utils.spark_utils import get_spark
from utils.logger import get_logger
from pyspark.sql.functions import col
from delta.tables import DeltaTable

spark = get_spark()
logger = get_logger()
logger.info("Starting Silver Layer")

In [ ]:
import json

with open("../../configs/entities.json") as f:
    config = json.load(f)

config = config['entities'][0]

In [ ]:
df = spark.read.table(config["bronze_table"])

df_clean = df.dropDuplicates(["coin", "timestamp"])

In [ ]:
target = config["silver_table"]

if spark.catalog.tableExists(target):

    delta_table = DeltaTable.forName(spark, target)

    delta_table.alias("t").merge(
        df_clean.alias("s"),
        "t.coin = s.coin AND t.ingestion_time = s.ingestion_time"
    ).whenNotMatchedInsertAll().execute()

else:
    df_clean.write.partitionBy("coin") \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target)

In [ ]:
logger.info("Silver Load Complete")